# Import Libaries

In [72]:
import kagglehub
import librosa
import pandas as pd

from pathlib import Path
from soundfile import LibsndfileError

# 0. Load data

In [73]:
path = kagglehub.dataset_download(
    "andradaolteanu/gtzan-dataset-music-genre-classification",
    output_dir="data"
)

print("Downloaded to:", path)
print(Path(path).resolve())

DATA_DIR = Path("data/Data/genres_original")

print("Path to dataset files:", DATA_DIR)

Downloaded to: data
D:\Programming_journey\portfolio\ai\ai-ml-portfolio\09-music-genre-classification\data
Path to dataset files: data\Data\genres_original


# 1. Data Inspection

## Genres

In [74]:
genres = sorted([folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()])

print("Number of genres:", len(genres))
print("Genres:", genres)

Number of genres: 10
Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


## Number of files in given genre

In [75]:
for genre in genres:
    files = list((DATA_DIR / genre).glob("*.wav"))
    print(f"{genre}: {len(files)}")

blues: 100
classical: 100
country: 100
disco: 100
hiphop: 100
jazz: 99
metal: 100
pop: 100
reggae: 100
rock: 100


## Create Data Frame

In [76]:
data = []

for genre in genres:
    for file in (DATA_DIR / genre).glob("*.wav"):
        data.append({
            "file_path": str(file),
            "genre": genre
        })

df = pd.DataFrame(data)

df.head()

,file_path,genre
0,data\Data\genres_original\blues\blues.00000.wav,blues
1,data\Data\genres_original\blues\blues.00001.wav,blues
2,data\Data\genres_original\blues\blues.00002.wav,blues
3,data\Data\genres_original\blues\blues.00003.wav,blues
4,data\Data\genres_original\blues\blues.00004.wav,blues


## Inspect All Audio Files in terms of Sample Rates and Duration

In [77]:
sample_rates = []
durations = []

for index, file_path in df["file_path"].copy().items():

    try:
        audio, sr = librosa.load(file_path, sr=None)

        df.loc[index, "sample_rate"] = sr
        df.loc[index, "duration"] = len(audio) / sr

    except LibsndfileError:
        print(f"Could not load: {file_path}")

        Path(file_path).unlink()
        df.drop(index, inplace=True)


df.reset_index(drop=True, inplace=True)

df.describe()

,sample_rate,duration
count,999.0,999.000000
mean,22050.0,30.024071
std,0.0,0.080951
min,22050.0,29.931973
25%,22050.0,30.000181
50%,22050.0,30.013333
75%,22050.0,30.013333
max,22050.0,30.648889


### Observation
All audio files have a consistent sample rate of **22,050 Hz** and an average duration of approximately **30 seconds**. The low standard deviation indicates that the audio files have very similar lengths.